## Embeddings

Hamare pichhle example mein humne `vocab_size` length wale high-dimensional bag-of-words vector par kaam kiya tha, aur low-dimensional positional representation vector ko explicitly sparse one-hot representation mein convert kiya tha. Yeh one-hot representation memory-efficient nahi hai. Saath hi, har word ko ek-doosre se independent maana jaata hai, isliye one-hot encoded vectors words ke beech ki semantic similarities ko express nahi kar paate.

Is unit mein hum **News AG** dataset ko aur detail mein explore karenge. Shuru karne ke liye, chalo data load karte hain aur pichhle unit se kuch definitions le lete hain.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ds_train, ds_test = tfds.load('ag_news_subset').values()

### Embedding kya hai?

**Embedding** ka idea yeh hai ki words ko low-dimensional dense vectors ke roop mein represent kiya jaaye, jo word ke semantic meaning ko reflect karte hain. Semantic word embeddings kaise banate hain, yeh hum baad mein dekhenge, lekin abhi ke liye embedding ko word vector ki dimensionality kam karne ka ek tareeka samajhiye.

Toh ek embedding layer ek word ko input mein leti hai aur specified `embedding_size` ka output vector produce karti hai. Ek tarah se yeh `Dense` layer jaisi hi hai, lekin one-hot encoded vector ko input lene ke bajaye yeh word number ko input ke roop mein le sakti hai.

Apne network ki pehli layer ke roop mein embedding layer use karke, hum bag-of-words se **embedding bag** model par switch kar sakte hain, jahan hum pehle apne text ke har word ko uski corresponding embedding mein convert karte hain, aur phir un sabhi embeddings par koi aggregate function calculate karte hain, jaise `sum`, `average` ya `max`.

![Ek image jo paanch sequence words ke liye embedding classifier dikhati hai.](../../../../../lessons/5-NLP/14-Embeddings/images/embedding-classifier-example.png)

Hamare classifier neural network mein yeh layers hain:

* `TextVectorization` layer, jo ek string ko input mein leti hai aur token numbers ka ek tensor produce karti hai. Hum ek suitable vocabulary size `vocab_size` specify karenge aur kam use hone wale words ko ignore karenge. Input shape 1 hogi, aur output shape $n$ hogi, kyunki humein $n$ tokens milenge, jinme se har ek mein 0 se `vocab_size` tak ka number hoga.
* `Embedding` layer, jo $n$ numbers leti hai aur har number ko ek specified length (hamare example mein 100) ke dense vector mein badal deti hai. Is tarah $n$ shape ke input tensor ko $n\times 100$ shape ke tensor mein convert kiya jaayega.
* Aggregation layer, jo is tensor ka pehle axis ke saath average leti hai, yaani yeh alag-alag words se related saare $n$ input tensors ka average calculate karegi. Is layer ko implement karne ke liye hum ek `Lambda` layer use karenge aur usme average calculate karne wala function pass karenge. Output ki shape 100 hogi, aur yeh poori input sequence ka numeric representation hoga.
* Aakhri `Dense` linear classifier.

In [ ]:
vocab_size = 30000
batch_size = 128

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,    
    keras.layers.Embedding(vocab_size,100),
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])
model.summary()

`summary` printout mein, **output shape** column ka pehla tensor dimension `None` minibatch size ko dikhata hai, aur doosra token sequence ki length ko. Minibatch mein saari token sequences ki length alag-alag hoti hai. Isse kaise handle karte hain, yeh hum agle section mein dekhenge.

Ab chalo network ko train karte hain:

In [ ]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

print("Training vectorizer")
vectorizer.adapt(ds_train.take(500).map(extract_text))

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

> **Note** ki hum data ke ek subset ke basis par vectorizer bana rahe hain. Yeh process ko tez karne ke liye kiya jaata hai, aur isse aisi situation ban sakti hai jab hamare text ke saare tokens vocabulary mein maujood nahi hote. Is case mein un tokens ko ignore kar diya jaayega, jisse accuracy thodi kam ho sakti hai. Lekin real life mein text ka ek subset aksar vocabulary ka achha approximation de deta hai.

### Variable sequence sizes ko handle karna

Chalo samajhte hain ki minibatches mein training kaise hoti hai. Upar diye gaye example mein input tensor ka dimension 1 hai, aur hum 128 length ka minibatch use karte hain, jisse tensor ki actual shape $128 \times 1$ ho jaati hai. Lekin har sentence mein tokens ki ginti alag hoti hai. Agar hum `TextVectorization` layer ko ek single input par apply karein, toh jitne tokens return hote hain woh alag-alag hote hain, yeh is par depend karta hai ki text ko kaise tokenize kiya gaya hai:

In [ ]:
print(vectorizer('Hello, world!'))
print(vectorizer('I am glad to meet you!'))

Lekin jab hum vectorizer ko kai sequences par apply karte hain, toh use ek rectangular shape ka tensor produce karna hota hai, isliye woh unused elements ko PAD token (jo hamare case mein zero hai) se bhar deta hai:

In [ ]:
vectorizer(['Hello, world!','I am glad to meet you!'])

Yahan hum embeddings dekh sakte hain:

In [ ]:
model.layers[1](vectorizer(['Hello, world!','I am glad to meet you!'])).numpy()

> **Note**: Padding ki matra kam karne ke liye, kuch cases mein samajhdaari yeh hoti hai ki dataset ki saari sequences ko unki length badhne ke order mein (ya zyada accurately, tokens ki ginti ke hisaab se) sort kar diya jaaye. Isse yeh ensure hoga ki har minibatch mein same length ki sequences hon.

## Semantic Embeddings: Word2Vec

Hamare pichhle example mein embedding layer ne words ko vector representation mein map karna seekha, lekin in representations mein semantic meaning nahi tha. Achha hoga agar hum aisa vector representation seekhein jisme similar words ya synonyms kisi vector distance (jaise Euclidean distance) ke hisaab se ek-doosre ke kareeb hon.

Iske liye humein apne embedding model ko [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) jaisi technique use karke text ke bade collection par pretrain karna hoga. Yeh do main architectures par based hai jo words ke distributed representation produce karne ke liye use hote hain:

 - **Continuous bag-of-words** (CBoW), jisme hum model ko aas-paas ke context se ek word predict karne ke liye train karte hain. Diye gaye ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ mein model ka goal $(W_{-2},W_{-1},W_1,W_2)$ se $W_0$ ko predict karna hai.
 - **Continuous skip-gram** CBoW ka ulta hai. Yeh model context words ki aas-paas ki window use karke current word ko predict karta hai.

CBoW tez hai, jabki skip-gram slow hai, lekin yeh kam use hone wale words ko behtar represent karta hai.

![Ek image jo words ko vectors mein convert karne ke liye CBoW aur skip-gram algorithms dikhati hai.](../../../../../lessons/5-NLP/14-Embeddings/images/example-algorithms-for-converting-words-to-vectors.png)

Google News dataset par pretrained Word2Vec embeddings ke saath experiment karne ke liye hum **gensim** library use kar sakte hain. Neeche hum 'neural' ke sabse similar words dhoondhte hain.

> **Note:** Jab aap pehli baar word vectors banate hain, toh unhe download hone mein kuch time lag sakta hai!

In [ ]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [ ]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

Hum word se vector embedding bhi nikaal sakte hain, jise classification model ki training mein use kiya ja sakta hai. Embedding mein 300 components hote hain, lekin yahan clarity ke liye hum vector ke sirf pehle 20 components dikha rahe hain:

In [ ]:
w2v['play'][:20]

Meaningful embeddings ki ek badi achhi baat yeh hai ki aap meaning ke basis par vector encoding ko modify kar sakte hain. Jaise, hum aisa word dhoondhne ko keh sakte hain jiska vector representation *king* aur *woman* words ke jitna ho sake kareeb ho, aur *man* word se jitna ho sake door ho:

In [ ]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

Upar diye gaye example mein kuch internal GenSym jaadu use hua hai, lekin basic logic actually kaafi simple hai. Embeddings ki ek interesting baat yeh hai ki aap embedding vectors par normal vector operations kar sakte hain, aur woh words ke **meanings** par operations ko reflect karega. Upar ke example ko vector operations ke roop mein express kiya ja sakta hai: hum **KING-MAN+WOMAN** ke corresponding vector calculate karte hain (related words ke vector representations par `+` aur `-` operations kiye jaate hain), aur phir dictionary mein us vector ke sabse kareebi word ko dhoondhte hain:

In [ ]:
# get the vector corresponding to kind-man+woman
qvec = w2v['king']-1.7*w2v['man']+1.7*w2v['woman']
# find the index of the closest embedding vector 
d = np.sum((w2v.vectors-qvec)**2,axis=1)
min_idx = np.argmin(d)
# find the corresponding word
w2v.index_to_key[min_idx]

> **NOTE**: Humein *man* aur *woman* vectors mein ek chhota coefficient add karna pada - ise hata kar dekhiye ki kya hota hai.

Sabse kareebi vector dhoondhne ke liye, hum TensorFlow ki technique use karte hain taaki hamare vector aur vocabulary ke saare vectors ke beech ki distances ka vector mil sake, aur phir `argmin` use karke minimum wale word ka index dhoondha ja sake.

Halaanki Word2Vec semantics express karne ka ek shaandaar tareeka lagta hai, lekin isme kai nuksaan bhi hain, jinme yeh shamil hain:

* CBoW aur skip-gram dono models **predictive embeddings** hain, aur yeh sirf local context ko dhyan mein rakhte hain. Word2Vec global context ka fayda nahi uthata.
* Word2Vec word ki **morphology** ko dhyan mein nahi rakhta, yaani is fact ko ki word ka meaning uske alag-alag parts, jaise root, par depend kar sakta hai.

**FastText** doosre restriction ko door karne ki koshish karta hai aur Word2Vec par based hokar har word aur uske andar paaye jaane wale character n-grams ke liye vector representations seekhta hai. In representations ki values ko har training step mein ek vector mein average kiya jaata hai. Halaanki isse pre-training mein extra computation jud jaata hai, yeh word embeddings ko sub-word information encode karne ke kaabil bana deta hai.

Ek aur method, **GloVe**, word embeddings ke liye alag approach leta hai, jo word-context matrix ke factorization par based hai. Sabse pehle yeh ek bada matrix banata hai jo alag-alag contexts mein words ki occurrences count karta hai, aur phir yeh is matrix ko low dimensions mein is tarah represent karne ki koshish karta hai ki reconstruction loss minimum ho.

gensim library in word embeddings ko support karti hai, aur aap upar diye gaye model loading code ko badal kar inke saath experiment kar sakte hain.

## Keras mein pretrained embeddings use karna

Hum upar diye gaye example ko modify kar sakte hain taaki apni embedding layer ke matrix ko Word2Vec jaisi semantic embeddings se pehle se bhar sakein. Pretrained embedding aur text corpus ki vocabularies shayad match nahi karengi, isliye humein ek ko choose karna hoga. Yahan hum do possible options explore karte hain: tokenizer vocabulary use karna, aur Word2Vec embeddings ki vocabulary use karna.

### Tokenizer vocabulary use karna

Tokenizer vocabulary use karte waqt, vocabulary ke kuch words ke liye Word2Vec embeddings available honge, aur kuch missing honge. Maan lo hamari vocabulary ka size `vocab_size` hai, aur Word2Vec embedding vector ki length `embed_size` hai, toh embedding layer ko `vocab_size`$\times$`embed_size` shape ke weight matrix se represent kiya jaayega. Hum is matrix ko vocabulary ke through jaakar bharenge:

In [ ]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

vocab = vectorizer.get_vocabulary()
W = np.zeros((vocab_size,embed_size))
print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab):
    try:
        W[i] = w2v.get_vector(w)
        found+=1
    except:
        # W[i] = np.random.normal(0.0,0.3,size=(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")

Jo words Word2Vec vocabulary mein maujood nahi hain, unhe hum zero ke roop mein chhod sakte hain, ya ek random vector generate kar sakte hain.

Ab hum pretrained weights ke saath ek embedding layer define kar sakte hain:

In [ ]:
emb = keras.layers.Embedding(vocab_size,embed_size,weights=[W],trainable=False)
model = keras.models.Sequential([
    vectorizer, emb,
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

> **Note**: Dhyan dijiye ki jab hum `Embedding` banate waqt `trainable=False` set karte hain, toh iska matlab hai ki hum Embedding layer ko dobara train nahi kar rahe. Isse accuracy thodi kam ho sakti hai, lekin yeh training ko tez kar deta hai.

### Embedding vocabulary use karna

Pichhle approach ke saath ek problem yeh hai ki TextVectorization aur Embedding mein use hui vocabularies alag-alag hain. Is problem ko solve karne ke liye hum neeche diye gaye solutions mein se ek use kar sakte hain:
* Apni vocabulary par Word2Vec model ko dobara train karein.
* Pretrained Word2Vec model ki vocabulary ke saath apna dataset load karein. Dataset load karte waqt use hone wali vocabularies specify ki ja sakti hain.

Doosra approach aasaan lagta hai, toh chalo ise implement karte hain. Sabse pehle, hum Word2Vec embedding se li gayi specified vocabulary ke saath ek `TextVectorization` layer banayenge:

In [ ]:
vocab = list(w2v.vocab.keys())
vectorizer = keras.layers.experimental.preprocessing.TextVectorization(input_shape=(1,))
vectorizer.set_vocabulary(vocab)

Gensim word embeddings library mein ek convenient function, `get_keras_embeddings`, hota hai, jo aapke liye automatically corresponding Keras embeddings layer bana dega.

In [ ]:
model = keras.models.Sequential([
    vectorizer, 
    w2v.get_keras_embedding(train_embeddings=False),
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(128),validation_data=ds_test.map(tupelize).batch(128),epochs=5)

Humein jo high accuracy nahi dikh rahi, uske kaaranon mein se ek yeh hai ki hamare dataset ke kuch words pretrained GloVe vocabulary mein nahi hain, aur isliye unhe ignore kar diya jaata hai. Isse door karne ke liye, hum apne dataset ke basis par apni khud ki embeddings train kar sakte hain.

## Contextual Embeddings

Word2Vec jaisi traditional pretrained embeddings ki ek main limitation yeh hai ki, bhale hi woh kisi word ka kuch meaning capture kar sakti hain, woh alag-alag meanings ke beech fark nahi kar sakti. Isse downstream models mein problems ho sakti hain.

Jaise, word 'play' ka in do sentences mein alag-alag meaning hai:
- Main theatre mein ek **play** dekhne gaya tha.
- John apne doston ke saath **play** karna chahta hai.

Humne jin pretrained embeddings ki baat ki, woh word 'play' ke dono meanings ko ek hi embedding mein represent karti hain. Is limitation ko door karne ke liye, humein **language model** par based embeddings banani hongi, jo bade text corpus par train hota hai aur *jaanta hai* ki words ko alag-alag contexts mein kaise jodha ja sakta hai. Contextual embeddings par baat karna is tutorial ke scope se bahar hai, lekin hum agle unit mein language models discuss karte waqt isse wapas aayenge.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->